# Lezione 5 — Encoding e scaling: dai dati alle feature

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 4 (usa
gli insiemi auditati). Alla fine di questa lezione il progetto avrà la sua
prima **matrice di feature**: numeri pronti a diventare tensori nella Fase 2.

**Cosa saprai fare alla fine:** trasformare qualunque tabella mista (testi,
categorie, numeri su scale diverse) in input numerici corretti per un
modello, senza introdurre ordini fittizi né leakage.

## Parte 1 — Teoria: i modelli calcolano, non leggono

Un modello è una funzione matematica: moltiplica, somma, confronta distanze.
Non sa cosa farsene di `"episodic"` o `"north"`. Ogni colonna non numerica va
**codificata**, e come la codifichi cambia cosa il modello può imparare.

**Encoding ordinale** — sostituire le categorie con interi
(`episodic→0, preference→1, semantic→2`). Compatto, ma **inventa un ordine
e delle distanze**: dice al modello che `semantic` è "il doppio" di
`preference` e che `episodic < preference`. Per categorie davvero ordinate
(taglie S/M/L, livelli basso/medio/alto) è perfetto; per categorie senza
ordine è una bugia strutturale che il modello prenderà sul serio.

**One-hot encoding** — una colonna binaria per categoria: `episodic`
diventa `(1,0,0)`, `preference` `(0,1,0)`, `semantic` `(0,0,1)`. Nessun
ordine inventato, nessuna distanza fittizia. Il costo: una colonna per
categoria — con 3 tipi va benissimo, con 10.000 città diventa ingestibile
(per quello esistono gli **embedding**, che sono esattamente il tema della
Fase 3 del corso: tieni il collegamento).

**Il target** (la cosa da predire) fa eccezione: al target serve solo un
intero per classe (0,1,2), perché non entra nei calcoli come feature — è la
risposta attesa.

## Parte 2 — Teoria: perché le scale contano

Molti modelli ragionano per **distanze** o per **gradienti**. Se una feature
vale ~50.000 (un reddito) e un'altra ~40 (un'età), la prima domina qualsiasi
distanza: l'età diventa invisibile, non perché conti meno, ma per un
incidente di unità di misura.

Le due ricette standard:

- **Standardizzazione** (z-score): `(x - media) / deviazione` → media 0,
  deviazione 1. La scelta di default.
- **Min-max**: riporta tutto in [0, 1]. Utile quando serve un range chiuso;
  sensibile agli outlier (un solo valore estremo schiaccia tutti gli altri).

E la regola che conosci già dalla Lezione 4, che qui diventa concreta:
media e deviazione si calcolano **sul solo train** e si applicano a
validation e test. Scalare tutto insieme è leakage da preprocessing.

Vediamo il problema delle scale con i numeri:

In [ ]:
import pandas as pd
import numpy as np

persone = pd.DataFrame({
    'eta':     [25,     45,     26],
    'reddito': [30_000, 90_000, 31_000],
}, index=['Anna', 'Bruno', 'Carla'])

def distanza(a, b):
    return float(np.sqrt(((persone.loc[a] - persone.loc[b]) ** 2).sum()))

print('Distanze senza scaling (domina il reddito):')
print(f'  Anna-Carla (simili davvero) : {distanza("Anna", "Carla"):8.0f}')
print(f'  Anna-Bruno (molto diversi)  : {distanza("Anna", "Bruno"):8.0f}')

media, dev = persone.mean(), persone.std()
scalate = (persone - media) / dev
def distanza_s(a, b):
    return float(np.sqrt(((scalate.loc[a] - scalate.loc[b]) ** 2).sum()))

print('\nDopo la standardizzazione (entrambe le feature contano):')
print(f'  Anna-Carla : {distanza_s("Anna", "Carla"):.2f}')
print(f'  Anna-Bruno : {distanza_s("Anna", "Bruno"):.2f}')

Senza scaling Anna e Carla — quasi identiche — distano quanto pesa la
differenza di reddito, e l'età è sparita dai calcoli. Dopo la
standardizzazione le distanze riflettono entrambe le dimensioni.

E il one-hot, in una riga di pandas:

In [ ]:
tipi = pd.DataFrame({'type': ['episodic', 'preference', 'semantic', 'episodic']})
pd.get_dummies(tipi, columns=['type'], dtype=int)

## Parte 3 — Esercizio guidato

Sui sensori: `station_id` è una categoria senza ordine, temperatura e
umidità sono numeri su scale diverse (gradi ~18, percentuali ~60).

Il tuo compito:

1. dividi le letture 70/30 (`train_test_split`, `random_state=0`) — qui non
   serve la divisione temporale, è un esercizio di trasformazione;
2. one-hot di `station_id` (attento: le colonne del test devono essere
   **le stesse** del train — usa `reindex`);
3. standardizza temperatura e umidità con **media e deviazione del train**;
4. verifica: le colonne scalate del train hanno media ~0 e deviazione ~1.

**Prova tu:**

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split

letture = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_missing_challenge.csv')
letture = letture.dropna().reset_index(drop=True)

# Scrivi qui il tuo tentativo (i 4 passi della consegna)

letture.head()

### Soluzione spiegata

- `get_dummies` sul train definisce il vocabolario delle colonne; il test
  viene **riallineato** a quel vocabolario con `reindex(..., fill_value=0)`:
  se in test comparisse una stazione mai vista, non inventerebbe colonne
  nuove (in produzione succede davvero);
- media e deviazione escono dal solo train (regola d'oro della Lezione 4)
  e si applicano a entrambi;
- gli assert finali verificano la definizione stessa di standardizzazione.

In [ ]:
train, test = train_test_split(letture, test_size=0.3, random_state=0)
train, test = train.copy(), test.copy()

X_train = pd.get_dummies(train[['station_id']], columns=['station_id'], dtype=int)
X_test = pd.get_dummies(test[['station_id']], columns=['station_id'], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

numeriche = ['temperature_c', 'humidity_pct']
media, dev = train[numeriche].mean(), train[numeriche].std()
X_train[numeriche] = (train[numeriche] - media) / dev
X_test[numeriche] = (test[numeriche] - media) / dev

assert X_train.columns.equals(X_test.columns)
assert X_train[numeriche].mean().abs().max() < 1e-9
assert (X_train[numeriche].std() - 1).abs().max() < 1e-9
X_train.head()

## Parte 4 — Il progetto: Memory AI Lab, passo 5

Il passo di oggi chiude la fase dati del progetto: dagli insiemi auditati
produciamo la **matrice di feature** che la Fase 2 trasformerà in tensori
per il primo classificatore (predire il `type` di una memoria).

Scelte, con le motivazioni:

- il **target** è `type` → interi per classe (mappa costruita sul train);
- dalle memorie estraiamo feature numeriche semplici: lunghezza del testo,
  numero di parole, `importance`, e i flag di audit delle Lezioni 1-2 (0/1
  — l'assenza è un segnale, ricordi?);
- tutte le numeriche vengono standardizzate con **statistiche del train**.

In [ ]:
processed = Path('..') / 'datasets' / 'processed'
insiemi = {nome: pd.read_csv(processed / f'memory_{nome}.csv')
           for nome in ['train', 'val', 'test']}

def estrai_feature(frame):
    out = pd.DataFrame(index=frame.index)
    testo = frame['text'].astype('string')
    out['text_length'] = testo.str.len()
    out['word_count'] = testo.str.split().str.len()
    out['importance'] = frame['importance']
    for flag in ['type_was_missing', 'importance_was_missing',
                 'importance_was_invalid_type', 'importance_was_outlier']:
        out[flag] = frame.get(flag, pd.Series(False, index=frame.index)).fillna(False).astype(int)
    return out

feature = {nome: estrai_feature(frame) for nome, frame in insiemi.items()}

classi = sorted(insiemi['train']['type'].unique())
mappa_classi = {classe: indice for indice, classe in enumerate(classi)}
print('Mappa del target:', mappa_classi)

In [ ]:
numeriche = ['text_length', 'word_count', 'importance']
media = feature['train'][numeriche].mean()
dev = feature['train'][numeriche].std()

for nome in feature:
    feature[nome][numeriche] = (feature[nome][numeriche] - media) / dev
    feature[nome]['target'] = insiemi[nome]['type'].map(mappa_classi)
    feature[nome].to_csv(processed / f'memory_features_{nome}.csv', index=False)

assert feature['train'][numeriche].mean().abs().max() < 1e-9
print('Matrici salvate:')
for nome, frame in feature.items():
    print(f'  memory_features_{nome}.csv: {frame.shape[0]} righe x {frame.shape[1]} colonne')
feature['train'].head()

Guarda l'ultima tabella: **tutto è numero**. Tre mesi di memorie testuali
sono diventati una matrice standardizzata con un target — il punto esatto in
cui finisce il data engineering e comincia il machine learning.

## Cosa hai imparato

- I modelli calcolano: le categorie vanno codificate, e **l'encoding
  ordinale inventa ordini** che il modello prende sul serio — one-hot per
  categorie senza ordine, embedding (Fase 3) quando sono troppe.
- Il **target** si mappa a interi: non è una feature.
- Le **scale** distorcono distanze e gradienti: standardizzazione come
  default, min-max quando serve un range chiuso.
- Ogni statistica di trasformazione viene **dal solo train** — l'encoding
  del test si riallinea al vocabolario del train (`reindex`).

## Quiz

1. Codifichi le città come `Milano=1, Roma=2, Napoli=3` e il modello impara
   che "Napoli = Milano × 3". Qual è l'errore e con quale encoding si evita?
2. Nel test compare una stazione mai vista nel train. Cosa fa la soluzione
   dell'esercizio, e perché è il comportamento giusto?
3. Perché i flag `was_missing` entrano nella matrice di feature invece di
   essere buttati dopo la pulizia?

<details>
<summary><b>Apri le risposte</b></summary>

1. L'encoding ordinale ha inventato un ordine e delle proporzioni tra
   categorie che non esistono. Con il one-hot ogni città è una dimensione
   indipendente e nessuna aritmetica fittizia è possibile.
2. `reindex(columns=..., fill_value=0)` produce tutte le colonne note con
   valore 0: la stazione nuova è "nessuna delle conosciute". È giusto
   perché in produzione le categorie mai viste arrivano davvero, e il
   modello deve riceverle in una forma prevista, non far esplodere la
   pipeline.
3. Perché l'assenza è informazione (Lezione 1): sapere che `importance` era
   mancante o fuori contratto può essere predittivo. Il flag trasporta quel
   segnale fino al modello.
</details>

## Fonti

- scikit-learn, *Preprocessing data* (standardizzazione, min-max, encoding):
  https://scikit-learn.org/stable/modules/preprocessing.html
- pandas, `get_dummies`:
  https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

## Prossimo passo del corso

La fase dati è completa: archivio pulito, split onesti, feature numeriche.
Nella **Fase 2** questi numeri diventano tensori e la matrice che hai appena
salvato addestra la prima rete neurale del Memory AI Lab.